In [1]:
import pandas as pd
import numpy as np

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from data import *

In [4]:
X_processed, y, emb_list, real_size, encoder = load_6()
X_processed[0, 5:10]

array([5.0, 2.0, 1.0, 1.0, 4.0], dtype=object)

In [5]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from utils import * # Assumes 'at_95' and other utils are available here

def run_ml_baselines_cv(X, y, model_type='lr', binary=True, n_seeds=100, splits=5):
    """
    model_type: 'lr' (Logistic Regression),
                'svm' (Support Vector Machine), or 'knn' (K-Nearest Neighbors)
    """
    all_seeds_metrics = []
    
    for SEED in tqdm(range(n_seeds), desc=f"Processing Seeds ({model_type.upper()})"):
        skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=SEED)
        for_stratify = y if binary else (y > 3.5).astype(int)
        
        fold_metrics = []
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, for_stratify), start=1):
            
            # --- Dataset Splits (Matches Original Code) ---
            X_train_full, X_test = X[train_idx], X[test_idx]
            y_train_full, y_test = y[train_idx], y[test_idx]
            
            val_stratify = y_train_full if binary else (y_train_full > 3.5).astype(int)
            train_sub_idx, val_idx = train_test_split(
                np.arange(len(X_train_full)),
                test_size=0.2,
                stratify=val_stratify,
                random_state=SEED
            )
            
            X_train, X_val = X_train_full[train_sub_idx], X_train_full[val_idx]
            y_train, y_val = y_train_full[train_sub_idx], y_train_full[val_idx]
            
            # --- Preprocessing ---
            categorical_cols = [0, 1, 2, 3, 4]
            numerical_cols = list(range(5, X.shape[1]))
            
            preprocessor = ColumnTransformer(
                transformers=[
                    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
                    ('num', StandardScaler(), numerical_cols)
                ]
            )
            
            X_train_proc = preprocessor.fit_transform(X_train)
            X_val_proc = preprocessor.transform(X_val) # Generated to maintain exact splits
            X_test_proc = preprocessor.transform(X_test)
            
            # --- Model Selection ---
            if model_type == 'lr':
                clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
            elif model_type == 'svm':
                clf = SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced')
            elif model_type == 'svm-lin':
                clf = SVC(kernel='linear', probability=True, random_state=SEED, class_weight='balanced')
            elif model_type == 'knn':
                clf = KNeighborsClassifier(n_neighbors=5, weights='distance')
            else:
                raise ValueError("model_type must be 'lr', 'svm', or 'knn'")
                
            # --- Model Training ---
            clf.fit(X_train_proc, y_train)
            
            # --- Evaluation ---
            if binary:
                logits_test = clf.predict_proba(X_test_proc)[:, 1] 
                y_pred = clf.predict(X_test_proc)
                
                a_test = (y_pred[y_test == 0] == y_test[y_test == 0]).mean() + 1e-20
                b_test = (y_pred[y_test == 1] == y_test[y_test == 1]).mean() + 1e-20
                c_test = 2 / ((1/a_test) + (1/b_test)) 
                
                auc_test = roc_auc_score(y_test, logits_test)
                
                sen_test = []
                spec_test = []
                try:
                    for t in [0.75, 0.8, 0.85, 0.9, 0.95]:
                        sen_, spec_ = at_95(logits_test, y_test, t=t)
                        sen_test.append(sen_)
                        spec_test.append(spec_)
                except NameError:
                    pass 
                    
                fold_metrics.append([a_test, b_test, c_test, auc_test, *spec_test, *sen_test])
                
        all_seeds_metrics.append(np.mean(fold_metrics, axis=0))
        
    final_metrics_mean = np.mean(all_seeds_metrics, axis=0)
    final_metrics_std = np.std(all_seeds_metrics, axis=0)
    
    # print("\n" + "="*50)
    print(f"FINAL RESULTS ACROSS 100 SEEDS: {model_type.upper()}")
    # print("="*50)
    if binary:
        print(f"Mean Accuracy Class 0 (a): {final_metrics_mean[0]:.4f} ± {final_metrics_std[0]:.4f}")
        print(f"Mean Accuracy Class 1 (b): {final_metrics_mean[1]:.4f} ± {final_metrics_std[1]:.4f}")
        print(f"Harmonic Mean (c):         {final_metrics_mean[2]:.4f} ± {final_metrics_std[2]:.4f}")
        print(f"ROC AUC:                   {final_metrics_mean[3]:.4f} ± {final_metrics_std[3]:.4f}")
        
    return final_metrics_mean, all_seeds_metrics

# --- Example Usage ---
# mean_metrics_lr, all_metrics_lr = run_ml_baselines_cv(X, y, model_type='lr')
# mean_metrics_svm, all_metrics_svm = run_ml_baselines_cv(X, y, model_type='svm')
# mean_metrics_knn, all_metrics_knn = run_ml_baselines_cv(X, y, model_type='knn')

In [6]:
_, __ = run_ml_baselines_cv(X_processed[:, 5:], y, binary=True, n_seeds=100, splits=5, model_type='lr')
_, __ = run_ml_baselines_cv(X_processed[:, 5:], y, binary=True, n_seeds=100, splits=5, model_type='svm')
_, __ = run_ml_baselines_cv(X_processed[:, 5:], y, binary=True, n_seeds=100, splits=5, model_type='svm-lin')
_, __ = run_ml_baselines_cv(X_processed[:, 5:], y, binary=True, n_seeds=100, splits=5, model_type='knn')

Processing Seeds (LR): 100%|█████████████████| 100/100 [00:14<00:00,  6.73it/s]


FINAL RESULTS ACROSS 100 SEEDS: LR
Mean Accuracy Class 0 (a): 0.6397 ± 0.0188
Mean Accuracy Class 1 (b): 0.7304 ± 0.0089
Harmonic Mean (c):         0.6797 ± 0.0115
ROC AUC:                   0.7417 ± 0.0099


Processing Seeds (SVM): 100%|████████████████| 100/100 [01:34<00:00,  1.06it/s]


FINAL RESULTS ACROSS 100 SEEDS: SVM
Mean Accuracy Class 0 (a): 0.5496 ± 0.0176
Mean Accuracy Class 1 (b): 0.8046 ± 0.0088
Harmonic Mean (c):         0.6502 ± 0.0123
ROC AUC:                   0.7566 ± 0.0067


Processing Seeds (SVM-LIN): 100%|████████████| 100/100 [08:39<00:00,  5.20s/it]


FINAL RESULTS ACROSS 100 SEEDS: SVM-LIN
Mean Accuracy Class 0 (a): 0.6582 ± 0.0209
Mean Accuracy Class 1 (b): 0.7077 ± 0.0104
Harmonic Mean (c):         0.6793 ± 0.0118
ROC AUC:                   0.7369 ± 0.0111


Processing Seeds (KNN): 100%|████████████████| 100/100 [00:19<00:00,  5.07it/s]

FINAL RESULTS ACROSS 100 SEEDS: KNN
Mean Accuracy Class 0 (a): 0.4929 ± 0.0219
Mean Accuracy Class 1 (b): 0.7317 ± 0.0159
Harmonic Mean (c):         0.5849 ± 0.0142
ROC AUC:                   0.6493 ± 0.0105


In [7]:
_, __ = run_ml_baselines_cv(X_processed[:, 5:], y, binary=True, n_seeds=100, splits=5, model_type='svm-lin')

Processing Seeds (SVM-LIN): 100%|████████████| 100/100 [08:38<00:00,  5.19s/it]

FINAL RESULTS ACROSS 100 SEEDS: SVM-LIN
Mean Accuracy Class 0 (a): 0.6582 ± 0.0209
Mean Accuracy Class 1 (b): 0.7077 ± 0.0104
Harmonic Mean (c):         0.6793 ± 0.0118
ROC AUC:                   0.7369 ± 0.0111


In [8]:
COHORTS = ['Autoimmune: Other', 'Cancer: Other', 'HIV', 'Healthy Control',
           'IBD', 'Multiple Myeloma', 'Transplant']

for cohort in COHORTS:
    X, y, emb_list, real_size, encoder = load_6()
    idx = X[:, 0] == cohort
    if (y[idx] == 0).sum() < 5:
        continue
    print(cohort)
    _, __ = run_ml_baselines_cv(X[idx, 5:], y[idx], binary=True, n_seeds=100, splits=5, model_type='lr')
    _, __ = run_ml_baselines_cv(X[idx, 5:], y[idx], binary=True, n_seeds=100, splits=5, model_type='svm')
    _, __ = run_ml_baselines_cv(X[idx, 5:], y[idx], binary=True, n_seeds=100, splits=5, model_type='knn')
    print('*****' * 15)

Autoimmune: Other


Processing Seeds (LR): 100%|█████████████████| 100/100 [00:07<00:00, 13.30it/s]


FINAL RESULTS ACROSS 100 SEEDS: LR
Mean Accuracy Class 0 (a): 0.4042 ± 0.0507
Mean Accuracy Class 1 (b): 0.6342 ± 0.0378
Harmonic Mean (c):         0.4769 ± 0.0413
ROC AUC:                   0.5107 ± 0.0321


Processing Seeds (SVM): 100%|████████████████| 100/100 [00:09<00:00, 10.32it/s]


FINAL RESULTS ACROSS 100 SEEDS: SVM
Mean Accuracy Class 0 (a): 0.2850 ± 0.0483
Mean Accuracy Class 1 (b): 0.7631 ± 0.0324
Harmonic Mean (c):         0.3990 ± 0.0511
ROC AUC:                   0.5147 ± 0.0374


...

FINAL RESULTS ACROSS 100 SEEDS: LR
Mean Accuracy Class 0 (a): 0.5404 ± 0.0459
Mean Accuracy Class 1 (b): 0.5176 ± 0.0494
Harmonic Mean (c):         0.5110 ± 0.0371
ROC AUC:                   0.5376 ± 0.0376


Processing Seeds (SVM): 100%|████████████████| 100/100 [00:07<00:00, 13.95it/s]


FINAL RESULTS ACROSS 100 SEEDS: SVM
Mean Accuracy Class 0 (a): 0.5507 ± 0.0678
Mean Accuracy Class 1 (b): 0.4564 ± 0.0564
Harmonic Mean (c):         0.4729 ± 0.0394
ROC AUC:                   0.4748 ± 0.0415


Processing Seeds (KNN): 100%|████████████████| 100/100 [00:07<00:00, 13.01it/s]

FINAL RESULTS ACROSS 100 SEEDS: KNN
Mean Accuracy Class 0 (a): 0.7214 ± 0.0502
Mean Accuracy Class 1 (b): 0.3238 ± 0.0498
Harmonic Mean (c):         0.4255 ± 0.0461
ROC AUC:                   0.5291 ± 0.0328
***************************************************************************


In [9]:
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from utils import * # Assumes 'at_95' is available

def run_ml_baselines_cv_by_cohort(X_features, y, cohort_labels, model_type='lr', binary=True, n_seeds=100, splits=5):
    """
    Trains on ALL data, evaluates by cohort.
    X_features: Fully processed numeric features (e.g., X_processed[:, 5:])
    y: Target labels
    cohort_labels: 1D array of cohort names (e.g., X_processed[:, 0])
    """
    unique_cohorts = np.unique(cohort_labels)
    all_seeds_metrics = {cohort: [] for cohort in unique_cohorts}
    
    for SEED in tqdm(range(n_seeds), desc=f"Processing Seeds ({model_type.upper()})"):
        skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=SEED)
        for_stratify = y if binary else (y > 3.5).astype(int)
        
        fold_metrics = {cohort: [] for cohort in unique_cohorts}
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X_features, for_stratify), start=1):
            
            # --- Dataset Splits ---
            X_train_full, X_test = X_features[train_idx], X_features[test_idx]
            y_train_full, y_test = y[train_idx], y[test_idx]
            
            # Grab the cohort identities just for this test fold
            cohorts_test = cohort_labels[test_idx]
            
            val_stratify = y_train_full if binary else (y_train_full > 3.5).astype(int)
            train_sub_idx, val_idx = train_test_split(
                np.arange(len(X_train_full)), test_size=0.2, stratify=val_stratify, random_state=SEED
            )
            
            # Note: We create validation sets here to keep the data splitting mathematically 
            # identical to your original loop, even if the baselines don't explicitly use X_val
            X_train, X_val = X_train_full[train_sub_idx], X_train_full[val_idx]
            y_train, y_val = y_train_full[train_sub_idx], y_train_full[val_idx]
            
            # --- Model Training ---
            if model_type == 'lr':
                clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
            elif model_type == 'svm':
                clf = SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced')
            elif model_type == 'knn':
                clf = KNeighborsClassifier(n_neighbors=5, weights='distance')
            
            # Train directly on the already-processed features
            clf.fit(X_train, y_train)
            
            # --- Global Prediction ---
            if binary:
                logits_test = clf.predict_proba(X_test)[:, 1] 
                y_pred = clf.predict(X_test)
                
                # --- Subgroup Evaluation (By Cohort) ---
                for cohort in unique_cohorts:
                    mask = (cohorts_test == cohort)
                    
                    if mask.sum() == 0:
                        continue 
                        
                    y_test_c = y_test[mask]
                    y_pred_c = y_pred[mask]
                    logits_test_c = logits_test[mask]
                    
                    class_0_correct = (y_pred_c[y_test_c == 0] == y_test_c[y_test_c == 0]).sum()
                    class_0_total = (y_test_c == 0).sum() + 1e-20
                    a_test = (class_0_correct / class_0_total) + 1e-20
                    
                    class_1_correct = (y_pred_c[y_test_c == 1] == y_test_c[y_test_c == 1]).sum()
                    class_1_total = (y_test_c == 1).sum() + 1e-20
                    b_test = (class_1_correct / class_1_total) + 1e-20
                    
                    c_test = 2 / ((1/a_test) + (1/b_test)) 
                    
                    if len(np.unique(y_test_c)) > 1:
                        auc_test = roc_auc_score(y_test_c, logits_test_c)
                    else:
                        auc_test = np.nan 
                    
                    fold_metrics[cohort].append([a_test, b_test, c_test, auc_test])
                
        # Average the 5 folds for this specific seed, for each cohort
        for cohort in unique_cohorts:
            if len(fold_metrics[cohort]) > 0:
                all_seeds_metrics[cohort].append(np.nanmean(fold_metrics[cohort], axis=0))
        
    # Final averaging across all 100 seeds
    print("\n" + "="*60)
    print(f"FINAL RESULTS BY COHORT ACROSS 100 SEEDS: {model_type.upper()}")
    print("="*60)
    
    final_cohort_results = {}
    for cohort in unique_cohorts:
        if len(all_seeds_metrics[cohort]) == 0:
            continue
            
        mean_metrics = np.nanmean(all_seeds_metrics[cohort], axis=0)
        std_metrics = np.nanstd(all_seeds_metrics[cohort], axis=0)
        final_cohort_results[cohort] = {'mean': mean_metrics, 'std': std_metrics}
        
        print(f"\n--- Cohort: {cohort} ---")
        print(f"Accuracy Class 0 (a): {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
        print(f"Accuracy Class 1 (b): {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
        print(f"Harmonic Mean (c):    {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")
        print(f"ROC AUC:              {mean_metrics[3]:.4f} ± {std_metrics[3]:.4f}")
        
    return final_cohort_results, all_seeds_metrics

In [10]:
# Pass only the numeric features to 'X_features', and the first column to 'cohort_labels'
X_processed, y, emb_list, real_size, encoder = load_6()
_, __ = run_ml_baselines_cv_by_cohort(
    X_features=X_processed[:, 5:], 
    y=y, 
    cohort_labels=X_processed[:, 0], 
    binary=True, 
    n_seeds=100, 
    splits=5, 
    model_type='lr'
)

# Repeat for the others

_, __ = run_ml_baselines_cv_by_cohort(X_processed[:, 5:], y, X_processed[:, 0], model_type='svm')
_, __ = run_ml_baselines_cv_by_cohort(X_processed[:, 5:], y, X_processed[:, 0], model_type='knn')

Processing Seeds (LR): 100%|█████████████████| 100/100 [00:57<00:00,  1.73it/s]



FINAL RESULTS BY COHORT ACROSS 100 SEEDS: LR

--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.3535 ± 0.0447
Accuracy Class 1 (b): 0.6901 ± 0.0281
Harmonic Mean (c):    0.4506 ± 0.0417
ROC AUC:              0.5287 ± 0.0208

--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.3955 ± 0.1504
Accuracy Class 1 (b): 0.7859 ± 0.0257
Harmonic Mean (c):    0.4029 ± 0.1349
ROC AUC:              0.6964 ± 0.0712

--- Cohort: HIV ---
Accuracy Class 0 (a): 0.5064 ± 0.1221
Accuracy Class 1 (b): 0.6911 ± 0.0451
Harmonic Mean (c):    0.5035 ± 0.1085
ROC AUC:              0.6331 ± 0.0585

--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.6044 ± 0.0941
Accuracy Class 1 (b): 0.7820 ± 0.0165
Harmonic Mean (c):    0.6442 ± 0.0761
ROC AUC:              0.7530 ± 0.0339

--- Cohort: IBD ---
Accuracy Class 0 (a): 0.4355 ± 0.1597
Accuracy Class 1 (b): 0.6179 ± 0.0700
Harmonic Mean (c):    0.3331 ± 0.1178
ROC AUC:              0.7405 ± 0.1174

--- Cohort: Multiple Myeloma ---
Accuracy Class 0 (a

Processing Seeds (SVM): 100%|████████████████| 100/100 [01:33<00:00,  1.08it/s]



FINAL RESULTS BY COHORT ACROSS 100 SEEDS: SVM

--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.3096 ± 0.0351
Accuracy Class 1 (b): 0.6242 ± 0.0287
Harmonic Mean (c):    0.3979 ± 0.0290
ROC AUC:              0.4571 ± 0.0143

--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.2798 ± 0.1183
Accuracy Class 1 (b): 0.8583 ± 0.0274
Harmonic Mean (c):    0.3062 ± 0.1099
ROC AUC:              0.6635 ± 0.0601

--- Cohort: HIV ---
Accuracy Class 0 (a): 0.5467 ± 0.0878
Accuracy Class 1 (b): 0.7424 ± 0.0364
Harmonic Mean (c):    0.5572 ± 0.0812
ROC AUC:              0.7361 ± 0.0484

--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.4786 ± 0.0673
Accuracy Class 1 (b): 0.7196 ± 0.0135
Harmonic Mean (c):    0.5321 ± 0.0660
ROC AUC:              0.6992 ± 0.0236

--- Cohort: IBD ---
Accuracy Class 0 (a): 0.4462 ± 0.1151
Accuracy Class 1 (b): 0.7375 ± 0.0384
Harmonic Mean (c):    0.3786 ± 0.0912
ROC AUC:              0.8732 ± 0.0649

--- Cohort: Multiple Myeloma ---
Accuracy Class 0 (

Processing Seeds (KNN): 100%|████████████████| 100/100 [00:16<00:00,  6.05it/s]


FINAL RESULTS BY COHORT ACROSS 100 SEEDS: KNN

--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.1357 ± 0.0305
Accuracy Class 1 (b): 0.8875 ± 0.0175
Harmonic Mean (c):    0.2222 ± 0.0462
ROC AUC:              0.5413 ± 0.0256

--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.1270 ± 0.0831
Accuracy Class 1 (b): 0.9483 ± 0.0183
Harmonic Mean (c):    0.1526 ± 0.0805
ROC AUC:              0.6785 ± 0.0814

--- Cohort: HIV ---
Accuracy Class 0 (a): 0.1414 ± 0.0853
Accuracy Class 1 (b): 0.9082 ± 0.0246
Harmonic Mean (c):    0.1891 ± 0.1014
ROC AUC:              0.5603 ± 0.0685

--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.3556 ± 0.0850
Accuracy Class 1 (b): 0.9190 ± 0.0083
Harmonic Mean (c):    0.4675 ± 0.0962
ROC AUC:              0.7241 ± 0.0364

--- Cohort: IBD ---
Accuracy Class 0 (a): 0.1683 ± 0.1302
Accuracy Class 1 (b): 0.9320 ± 0.0387
Harmonic Mean (c):    0.1718 ± 0.1289
ROC AUC:              0.8537 ± 0.1009

--- Cohort: Multiple Myeloma ---
Accuracy Class 0 (

In [11]:
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from utils import * # Assumes 'at_95' is available

def run_ml_baselines_transfer(X_train_feats, y_train, pid_train, 
                              X_test_feats, y_test, pid_test, 
                              model_type='lr', binary=True, n_seeds=100, splits=5):
    """
    Trains on folds of Dataset 6, evaluates on the mapped patients in Dataset 2.
    """
    all_seeds_metrics = []
    
    for SEED in tqdm(range(n_seeds), desc=f"Processing Seeds ({model_type.upper()})"):
        skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=SEED)
        for_stratify = y_train if binary else (y_train > 3.5).astype(int)
        
        fold_metrics = []
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X_train_feats, for_stratify), start=1):
            
            # --- Dataset Splits ---
            # 1. Build Training Set entirely from Train Source (Dataset 6)
            X_train_fold = X_train_feats[train_idx]
            y_train_fold = y_train[train_idx]
            
            # 2. Build Testing Set from Test Source (Dataset 2) by mapping PIDs
            test_pids = pid_train[test_idx]
            test_mask = np.isin(pid_test, test_pids)
            
            # If no matching patients are found in Dataset 2 for this test group, safely skip
            if test_mask.sum() == 0:
                continue
                
            X_test_fold = X_test_feats[test_mask]
            y_test_fold = y_test[test_mask]
            
            # --- Model Training ---
            if model_type == 'lr':
                clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
            
            elif model_type == 'svm':
                clf = SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced')
            elif model_type == 'svm-lin':
                clf = SVC(kernel='linear', probability=True, random_state=SEED, class_weight='balanced')
            elif model_type == 'knn':
                clf = KNeighborsClassifier(n_neighbors=5, weights='distance')
            
            # .astype(float) prevents errors if the arrays retained an 'object' datatype from earlier textual columns
            clf.fit(X_train_fold.astype(float), y_train_fold.astype(float))
            
            # --- Evaluation ---
            if binary:
                logits_test = clf.predict_proba(X_test_fold.astype(float))[:, 1] 
                y_pred = clf.predict(X_test_fold.astype(float))
                
                # Prevent division by zero
                class_0_correct = (y_pred[y_test_fold == 0] == y_test_fold[y_test_fold == 0]).sum()
                class_0_total = (y_test_fold == 0).sum() + 1e-20
                a_test = (class_0_correct / class_0_total) + 1e-20
                
                class_1_correct = (y_pred[y_test_fold == 1] == y_test_fold[y_test_fold == 1]).sum()
                class_1_total = (y_test_fold == 1).sum() + 1e-20
                b_test = (class_1_correct / class_1_total) + 1e-20
                
                c_test = 2 / ((1/a_test) + (1/b_test)) 
                
                if len(np.unique(y_test_fold)) > 1:
                    auc_test = roc_auc_score(y_test_fold, logits_test)
                else:
                    auc_test = np.nan
                    
                sen_test = []
                spec_test = []
                try:
                    for t in [0.75, 0.8, 0.85, 0.9, 0.95]:
                        sen_, spec_ = at_95(logits_test, y_test_fold, t=t)
                        sen_test.append(sen_)
                        spec_test.append(spec_)
                except NameError:
                    pass 
                    
                fold_metrics.append([a_test, b_test, c_test, auc_test, *spec_test, *sen_test])
                
        if len(fold_metrics) > 0:
            all_seeds_metrics.append(np.nanmean(fold_metrics, axis=0))
        
    final_metrics_mean = np.nanmean(all_seeds_metrics, axis=0)
    final_metrics_std = np.nanstd(all_seeds_metrics, axis=0)
    
    print("\n" + "="*60)
    print(f"FINAL RESULTS: {model_type.upper()} (TRAIN: DS6 -> TEST: DS2)")
    print("="*60)
    if binary:
        print(f"Accuracy Class 0 (a): {final_metrics_mean[0]:.4f} ± {final_metrics_std[0]:.4f}")
        print(f"Accuracy Class 1 (b): {final_metrics_mean[1]:.4f} ± {final_metrics_std[1]:.4f}")
        print(f"Harmonic Mean (c):    {final_metrics_mean[2]:.4f} ± {final_metrics_std[2]:.4f}")
        print(f"ROC AUC:              {final_metrics_mean[3]:.4f} ± {final_metrics_std[3]:.4f}")
        
    return final_metrics_mean, all_seeds_metrics

In [12]:
# Assuming your X6 and X2 still contain text in columns 0-4
X6, y6 , pid6, emb_list6, real_size6, encoder6 = load_6(binary=True, include_pid=True)

X2, y2 , pid2, emb_list2, real_size2, encoder2 = load_2(binary=True, include_pid=True)

# size = real_size6 if use_cat else real_size6 - 1

en = {}
for k_cohort, v_cohort in (X6[:, [0, 5]]):
    en[k_cohort] = v_cohort
X_train_features = X6[:, 5:]
X_test_features = X2[:, 5:]

_, __ = run_ml_baselines_transfer(
    X_train_feats=X_train_features, y_train=y6, pid_train=pid6,
    X_test_feats=X_test_features, y_test=y2, pid_test=pid2,
    model_type='lr', n_seeds=100, splits=5
)

_, __ = run_ml_baselines_transfer(X_train_features, y6, pid6, X_test_features, y2, pid2, model_type='svm')
# _, __ = run_ml_baselines_transfer(X_train_features, y6, pid6, X_test_features, y2, pid2, model_type='svm-lin')
_, __ = run_ml_baselines_transfer(X_train_features, y6, pid6, X_test_features, y2, pid2, model_type='knn')

Processing Seeds (LR): 100%|█████████████████| 100/100 [01:08<00:00,  1.45it/s]



FINAL RESULTS: LR (TRAIN: DS6 -> TEST: DS2)
Accuracy Class 0 (a): 0.6108 ± 0.0227
Accuracy Class 1 (b): 0.6627 ± 0.0091
Harmonic Mean (c):    0.6319 ± 0.0128
ROC AUC:              0.6737 ± 0.0105


Processing Seeds (SVM): 100%|████████████████| 100/100 [02:13<00:00,  1.33s/it]



FINAL RESULTS: SVM (TRAIN: DS6 -> TEST: DS2)
Accuracy Class 0 (a): 0.6759 ± 0.0125
Accuracy Class 1 (b): 0.6414 ± 0.0061
Harmonic Mean (c):    0.6550 ± 0.0074
ROC AUC:              0.6988 ± 0.0044


Processing Seeds (KNN): 100%|████████████████| 100/100 [00:15<00:00,  6.38it/s]


FINAL RESULTS: KNN (TRAIN: DS6 -> TEST: DS2)
Accuracy Class 0 (a): 0.3311 ± 0.0211
Accuracy Class 1 (b): 0.8667 ± 0.0066
Harmonic Mean (c):    0.4735 ± 0.0226
ROC AUC:              0.6592 ± 0.0098


In [13]:
_, __ = run_ml_baselines_transfer(X_train_features, y6, pid6, X_test_features, y2, pid2, model_type='svm-lin')

Processing Seeds (SVM-LIN): 100%|████████████| 100/100 [48:14<00:00, 28.94s/it]


FINAL RESULTS: SVM-LIN (TRAIN: DS6 -> TEST: DS2)
Accuracy Class 0 (a): 0.6496 ± 0.0247
Accuracy Class 1 (b): 0.6266 ± 0.0118
Harmonic Mean (c):    0.6342 ± 0.0136
ROC AUC:              0.6700 ± 0.0123


In [14]:
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from utils import * # Assumes 'at_95' is available

def run_ml_baselines_transfer_by_cohort(X_train_feats, y_train, pid_train, 
                                        X_test_feats, y_test, pid_test, cohort_test,
                                        model_type='lr', binary=True, n_seeds=100, splits=5):
    """
    Trains on DS6 folds, evaluates on mapped patients in DS2, and groups stats by cohort.
    cohort_test: 1D array of cohort names matching the rows of X_test_feats (e.g., X2[:, 0])
    """
    unique_cohorts = np.unique(cohort_test)
    all_seeds_metrics = {cohort: [] for cohort in unique_cohorts}
    
    for SEED in tqdm(range(n_seeds), desc=f"Processing Seeds ({model_type.upper()})"):
        skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=SEED)
        for_stratify = y_train if binary else (y_train > 3.5).astype(int)
        
        fold_metrics = {cohort: [] for cohort in unique_cohorts}
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X_train_feats, for_stratify), start=1):
            
            # --- Dataset Splits ---
            X_train_fold = X_train_feats[train_idx]
            y_train_fold = y_train[train_idx]
            
            # Map PIDs to build the test set for DS2
            test_pids = pid_train[test_idx]
            test_mask = np.isin(pid_test, test_pids)
            
            if test_mask.sum() == 0:
                continue
                
            X_test_fold = X_test_feats[test_mask]
            y_test_fold = y_test[test_mask]
            
            # Grab the cohort labels specific to this test group
            cohorts_test_fold = cohort_test[test_mask]
            
            # --- Model Training ---
            if model_type == 'lr':
                clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
            
            elif model_type == 'svm':
                clf = SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced')
            elif model_type == 'knn':
                clf = KNeighborsClassifier(n_neighbors=5, weights='distance')
            
            clf.fit(X_train_fold.astype(float), y_train_fold.astype(float))
            
            # --- Global Prediction on the Test Fold ---
            if binary:
                logits_test = clf.predict_proba(X_test_fold.astype(float))[:, 1] 
                y_pred = clf.predict(X_test_fold.astype(float))
                
                # --- Subgroup Evaluation (By Cohort in DS2) ---
                for cohort in unique_cohorts:
                    mask_c = (cohorts_test_fold == cohort)
                    
                    if mask_c.sum() == 0:
                        continue 
                        
                    y_test_c = y_test_fold[mask_c]
                    y_pred_c = y_pred[mask_c]
                    logits_test_c = logits_test[mask_c]
                    
                    class_0_correct = (y_pred_c[y_test_c == 0] == y_test_c[y_test_c == 0]).sum()
                    class_0_total = (y_test_c == 0).sum() + 1e-20
                    a_test = (class_0_correct / class_0_total) + 1e-20
                    
                    class_1_correct = (y_pred_c[y_test_c == 1] == y_test_c[y_test_c == 1]).sum()
                    class_1_total = (y_test_c == 1).sum() + 1e-20
                    b_test = (class_1_correct / class_1_total) + 1e-20
                    
                    c_test = 2 / ((1/a_test) + (1/b_test)) 
                    
                    # Only calculate AUC if both classes exist in this cohort for this fold
                    if len(np.unique(y_test_c)) > 1:
                        auc_test = roc_auc_score(y_test_c, logits_test_c)
                    else:
                        auc_test = np.nan 
                    
                    fold_metrics[cohort].append([a_test, b_test, c_test, auc_test])
                
        # Average the 5 folds for this specific seed, for each cohort
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning) # Ignore 'mean of empty slice' warnings
            for cohort in unique_cohorts:
                if len(fold_metrics[cohort]) > 0:
                    all_seeds_metrics[cohort].append(np.nanmean(fold_metrics[cohort], axis=0))
        
    # Final averaging across all 100 seeds, grouped by cohort
    print("\n" + "="*70)
    print(f"FINAL RESULTS BY COHORT: {model_type.upper()} (TRAIN: DS6 -> TEST: DS2)")
    print("="*70)
    
    final_cohort_results = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        for cohort in unique_cohorts:
            if len(all_seeds_metrics[cohort]) == 0:
                print(f"\n--- Cohort: {cohort} ---")
                print("No data evaluated for this cohort across any seeds/folds.")
                continue
                
            mean_metrics = np.nanmean(all_seeds_metrics[cohort], axis=0)
            std_metrics = np.nanstd(all_seeds_metrics[cohort], axis=0)
            final_cohort_results[cohort] = {'mean': mean_metrics, 'std': std_metrics}
            
            print(f"\n--- Cohort: {cohort} ---")
            print(f"Accuracy Class 0 (a): {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
            print(f"Accuracy Class 1 (b): {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
            print(f"Harmonic Mean (c):    {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")
            print(f"ROC AUC:              {mean_metrics[3]:.4f} ± {std_metrics[3]:.4f}")
        
    return final_cohort_results, all_seeds_metrics


# Assuming X6 and X2 still contain text in columns 0-4
X_train_features = X6[:, 5:]
X_test_features = X2[:, 5:]

# The cohort labels for Dataset 2 are in the first column of X2
cohort_labels_ds2 = X2[:, 0]

_, __ = run_ml_baselines_transfer_by_cohort(
    X_train_feats=X_train_features, y_train=y6, pid_train=pid6,
    X_test_feats=X_test_features,   y_test=y2,  pid_test=pid2,
    cohort_test=cohort_labels_ds2,
    model_type='lr', n_seeds=100, splits=5
)

# Repeat for the others
_, __ = run_ml_baselines_transfer_by_cohort(X_train_features, y6, pid6, X_test_features, y2, pid2, cohort_labels_ds2, model_type='svm')
_, __ = run_ml_baselines_transfer_by_cohort(X_train_features, y6, pid6, X_test_features, y2, pid2, cohort_labels_ds2, model_type='knn')

Processing Seeds (LR): 100%|█████████████████| 100/100 [01:10<00:00,  1.41it/s]



FINAL RESULTS BY COHORT: LR (TRAIN: DS6 -> TEST: DS2)

--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.2517 ± 0.0642
Accuracy Class 1 (b): 0.7554 ± 0.0329
Harmonic Mean (c):    0.3384 ± 0.0715
ROC AUC:              0.5143 ± 0.0400

--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.3677 ± 0.1373
Accuracy Class 1 (b): 0.7675 ± 0.0271
Harmonic Mean (c):    0.3511 ± 0.1203
ROC AUC:              0.7177 ± 0.0771

--- Cohort: HIV ---
Accuracy Class 0 (a): 0.4689 ± 0.1240
Accuracy Class 1 (b): 0.6886 ± 0.0361
Harmonic Mean (c):    0.4690 ± 0.1045
ROC AUC:              0.6319 ± 0.0615

--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.5646 ± 0.1028
Accuracy Class 1 (b): 0.7628 ± 0.0120
Harmonic Mean (c):    0.6031 ± 0.0861
ROC AUC:              0.7162 ± 0.0386

--- Cohort: IBD ---
Accuracy Class 0 (a): 0.3733 ± 0.1257
Accuracy Class 1 (b): 0.6416 ± 0.0622
Harmonic Mean (c):    0.2932 ± 0.1043
ROC AUC:              0.7533 ± 0.1323

--- Cohort: Multiple Myeloma ---
Accuracy C

Processing Seeds (SVM): 100%|████████████████| 100/100 [02:14<00:00,  1.34s/it]



FINAL RESULTS BY COHORT: SVM (TRAIN: DS6 -> TEST: DS2)

--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.3048 ± 0.0510
Accuracy Class 1 (b): 0.7432 ± 0.0233
Harmonic Mean (c):    0.3908 ± 0.0566
ROC AUC:              0.5017 ± 0.0321

--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.1653 ± 0.0824
Accuracy Class 1 (b): 0.8656 ± 0.0185
Harmonic Mean (c):    0.1810 ± 0.0738
ROC AUC:              0.6683 ± 0.0628

--- Cohort: HIV ---
Accuracy Class 0 (a): 0.4951 ± 0.0923
Accuracy Class 1 (b): 0.7736 ± 0.0254
Harmonic Mean (c):    0.5276 ± 0.0792
ROC AUC:              0.6694 ± 0.0528

--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.3721 ± 0.0715
Accuracy Class 1 (b): 0.7315 ± 0.0073
Harmonic Mean (c):    0.4397 ± 0.0722
ROC AUC:              0.6453 ± 0.0295

--- Cohort: IBD ---
Accuracy Class 0 (a): 0.3230 ± 0.0855
Accuracy Class 1 (b): 0.6789 ± 0.0553
Harmonic Mean (c):    0.2608 ± 0.0766
ROC AUC:              0.8061 ± 0.0888

--- Cohort: Multiple Myeloma ---
Accuracy 

Processing Seeds (KNN): 100%|████████████████| 100/100 [00:17<00:00,  5.72it/s]


FINAL RESULTS BY COHORT: KNN (TRAIN: DS6 -> TEST: DS2)

--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.0898 ± 0.0443
Accuracy Class 1 (b): 0.9177 ± 0.0205
Harmonic Mean (c):    0.1407 ± 0.0642
ROC AUC:              0.4958 ± 0.0416

--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.1413 ± 0.1123
Accuracy Class 1 (b): 0.9352 ± 0.0168
Harmonic Mean (c):    0.1603 ± 0.1178
ROC AUC:              0.7911 ± 0.0699

--- Cohort: HIV ---
Accuracy Class 0 (a): 0.1770 ± 0.0828
Accuracy Class 1 (b): 0.9065 ± 0.0230
Harmonic Mean (c):    0.2329 ± 0.0944
ROC AUC:              0.5261 ± 0.0630

--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.1507 ± 0.0506
Accuracy Class 1 (b): 0.9282 ± 0.0065
Harmonic Mean (c):    0.2239 ± 0.0663
ROC AUC:              0.6052 ± 0.0383

--- Cohort: IBD ---
Accuracy Class 0 (a): 0.1360 ± 0.1136
Accuracy Class 1 (b): 0.9168 ± 0.0385
Harmonic Mean (c):    0.1367 ± 0.1116
ROC AUC:              0.6878 ± 0.0888

--- Cohort: Multiple Myeloma ---
Accuracy 

In [15]:
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from utils import * # Assumes 'at_95' is available

def run_transfer_cohort_to_cohort(X_train_feats, y_train, pid_train, cohort_train,
                                  X_test_feats, y_test, pid_test, cohort_test,
                                  model_type='lr', binary=True, n_seeds=100, splits=5):
    """
    Filters datasets by cohort first. 
    Trains strictly on Cohort X from DS6, evaluates strictly on mapped patients in Cohort X from DS2.
    """
    unique_cohorts = np.unique(cohort_train)
    final_results = {}
    
    print("\n" + "="*70)
    print(f"STARTING PURE COHORT-TO-COHORT EVALUATION: {model_type.upper()}")
    print("="*70)

    for cohort in unique_cohorts:
        # 1. Isolate both datasets to the current cohort
        mask_train = (cohort_train == cohort)
        mask_test = (cohort_test == cohort)
        
        X_tr_c, y_tr_c, pid_tr_c = X_train_feats[mask_train], y_train[mask_train], pid_train[mask_train]
        X_te_c, y_te_c, pid_te_c = X_test_feats[mask_test], y_test[mask_test], pid_test[mask_test]
        
        # 2. Safety Check: Ensure enough data exists to run splits
        # Matches your original logic of needing at least 5 majority class samples
        if len(y_tr_c) == 0 or len(y_te_c) == 0 or (y_tr_c == 0).sum() < splits:
            print(f"\n--- Cohort: {cohort} --- [SKIPPED: Insufficient Data]")
            continue
            
        all_seeds_metrics = []
        
        # 3. Cross-Validation strictly on this cohort's subset
        for SEED in tqdm(range(n_seeds), desc=f"{cohort} ({model_type.upper()})", leave=False):
            skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=SEED)
            for_stratify = y_tr_c if binary else (y_tr_c > 3.5).astype(int)
            
            fold_metrics = []
            
            for fold, (train_idx, test_idx) in enumerate(skf.split(X_tr_c, for_stratify), start=1):
                
                # --- Train Split (DS6 Cohort Subset) ---
                X_train_fold = X_tr_c[train_idx]
                y_train_fold = y_tr_c[train_idx]
                
                # --- Test Split (Mapped to DS2 Cohort Subset) ---
                test_pids = pid_tr_c[test_idx]
                test_mask = np.isin(pid_te_c, test_pids)
                
                if test_mask.sum() == 0:
                    continue
                    
                X_test_fold = X_te_c[test_mask]
                y_test_fold = y_te_c[test_mask]
                
                # --- Model Training ---
                if model_type == 'lr':
                    clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
                
                elif model_type == 'svm':
                    clf = SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced')
                elif model_type == 'knn':
                    clf = KNeighborsClassifier(n_neighbors=5, weights='distance')
                
                clf.fit(X_train_fold.astype(float), y_train_fold.astype(float))
                
                # --- Evaluation ---
                if binary:
                    logits_test = clf.predict_proba(X_test_fold.astype(float))[:, 1] 
                    y_pred = clf.predict(X_test_fold.astype(float))
                    
                    class_0_correct = (y_pred[y_test_fold == 0] == y_test_fold[y_test_fold == 0]).sum()
                    class_0_total = (y_test_fold == 0).sum() + 1e-20
                    a_test = (class_0_correct / class_0_total) + 1e-20
                    
                    class_1_correct = (y_pred[y_test_fold == 1] == y_test_fold[y_test_fold == 1]).sum()
                    class_1_total = (y_test_fold == 1).sum() + 1e-20
                    b_test = (class_1_correct / class_1_total) + 1e-20
                    
                    c_test = 2 / ((1/a_test) + (1/b_test)) 
                    
                    if len(np.unique(y_test_fold)) > 1:
                        auc_test = roc_auc_score(y_test_fold, logits_test)
                    else:
                        auc_test = np.nan 
                        
                    fold_metrics.append([a_test, b_test, c_test, auc_test])
                    
            if len(fold_metrics) > 0:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    all_seeds_metrics.append(np.nanmean(fold_metrics, axis=0))
            
        # 4. Final Aggregation for this specific Cohort
        if len(all_seeds_metrics) > 0:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                mean_metrics = np.nanmean(all_seeds_metrics, axis=0)
                std_metrics = np.nanstd(all_seeds_metrics, axis=0)
                
            final_results[cohort] = {'mean': mean_metrics, 'std': std_metrics, 'raw': all_seeds_metrics}
            
            print(f"\n--- Cohort: {cohort} ---")
            print(f"Accuracy Class 0 (a): {mean_metrics[0]:.4f} ± {std_metrics[0]:.4f}")
            print(f"Accuracy Class 1 (b): {mean_metrics[1]:.4f} ± {std_metrics[1]:.4f}")
            print(f"Harmonic Mean (c):    {mean_metrics[2]:.4f} ± {std_metrics[2]:.4f}")
            print(f"ROC AUC:              {mean_metrics[3]:.4f} ± {std_metrics[3]:.4f}")
        else:
            print(f"\n--- Cohort: {cohort} --- [FAILED: No overlapping test sets]")

    return final_results


# Strip textual data
X_train_features = X6[:, 5:]
X_test_features = X2[:, 5:]

# Get cohort label arrays
cohort_labels_ds6 = X6[:, 0]
cohort_labels_ds2 = X2[:, 0]

# Run it! 
# (This loop handles all cohorts automatically, so you don't need a custom 'for' loop)
results_lr = run_transfer_cohort_to_cohort(
    X_train_feats=X_train_features, y_train=y6, pid_train=pid6, cohort_train=cohort_labels_ds6,
    X_test_feats=X_test_features,   y_test=y2,  pid_test=pid2,  cohort_test=cohort_labels_ds2,
    model_type='lr'
)

# Repeat for the other baselines

results_svm = run_transfer_cohort_to_cohort(X_train_features, y6, pid6, cohort_labels_ds6, X_test_features, y2, pid2, cohort_labels_ds2, model_type='svm')
results_knn = run_transfer_cohort_to_cohort(X_train_features, y6, pid6, cohort_labels_ds6, X_test_features, y2, pid2, cohort_labels_ds2, model_type='knn')


STARTING PURE COHORT-TO-COHORT EVALUATION: LR



--- Cohort: Autoimmune: Other ---
Accuracy Class 0 (a): 0.4661 ± 0.0697
Accuracy Class 1 (b): 0.5772 ± 0.0436
Harmonic Mean (c):    0.4781 ± 0.0561
ROC AUC:              0.5059 ± 0.0403



--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.1300 ± 0.0975
Accuracy Class 1 (b): 0.8942 ± 0.0273
Harmonic Mean (c):    0.1335 ± 0.0889
ROC AUC:              0.4311 ± 0.0942



--- Cohort: HIV ---
Accuracy Class 0 (a): 0.2647 ± 0.0730
Accuracy Class 1 (b): 0.7975 ± 0.0338
Harmonic Mean (c):    0.3237 ± 0.0899
ROC AUC:              0.5179 ± 0.0428



--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.2699 ± 0.0488
Accuracy Class 1 (b): 0.9322 ± 0.0091
Harmonic Mean (c):    0.3749 ± 0.0601
ROC AUC:              0.7392 ± 0.0226

--- Cohort: IBD --- [SKIPPED: Insufficient Data]


...


--- Cohort: Cancer: Other ---
Accuracy Class 0 (a): 0.0000 ± 0.0000
Accuracy Class 1 (b): 0.9856 ± 0.0093
Harmonic Mean (c):    0.0000 ± 0.0000
ROC AUC:              0.4589 ± 0.0572



--- Cohort: HIV ---
Accuracy Class 0 (a): 0.1800 ± 0.0596
Accuracy Class 1 (b): 0.9258 ± 0.0273
Harmonic Mean (c):    0.2452 ± 0.0812
ROC AUC:              0.5178 ± 0.0463



--- Cohort: Healthy Control ---
Accuracy Class 0 (a): 0.0094 ± 0.0210
Accuracy Class 1 (b): 0.9903 ± 0.0032
Harmonic Mean (c):    0.0150 ± 0.0332
ROC AUC:              0.7127 ± 0.0263

--- Cohort: IBD --- [SKIPPED: Insufficient Data]



--- Cohort: Multiple Myeloma ---
Accuracy Class 0 (a): 0.7238 ± 0.0491
Accuracy Class 1 (b): 0.3378 ± 0.0347
Harmonic Mean (c):    0.4421 ± 0.0339
ROC AUC:              0.5480 ± 0.0316



--- Cohort: Transplant ---
Accuracy Class 0 (a): 0.6000 ± 0.0471
Accuracy Class 1 (b): 0.4893 ± 0.0419
Harmonic Mean (c):    0.5127 ± 0.0385
ROC AUC:              0.5635 ± 0.0344


In [16]:
# df['PID']
df1 = pd.read_csv('Leidos_data_98_Abs_with_clinical_features_v3.csv')
df2 = pd.read_csv('Leidos_data_98_Abs_with_clinical_features_v2.csv')
(df1['PID'] == df2['PID']).astype(float).mean()

1.0

In [17]:
list(df2.columns)

['Sample_ID',
 'PID',
 'Cohort',
 'V0_Group',
 'B0_Group',
 'Vaccine_Response_raw',
 'Vaccine_Response_class',
 'Age',
 'Age_Group',
 'Age_Group_1',
 'Sex',
 'Race',
 'A.calcoaceticus LysM',
 'A.prevotii C40',
 'A4-Fla2',
 'AiV1 VP3',
 'C.difficile SH3',
 'C.koseri Flg',
 'CVA10 VP1',
 'CVA16 PolyP',
 'CVA2 PolyP',
 'CVA22 PolyP',
 'CVA24 PolyP',
 'CVA4 PolyP',
 'CVA5 PolyP',
 'CVA6 PolyP',
 'CVA9 PolyP',
 'CVB1 PolyP',
 'CVB2 PolyP',
 'CVB3 PolyP',
 'CVB4 VP1',
 'CVB5 PolyP',
 'CVB6 PolyP',
 'E.coli FliC',
 'EBV BFRF3',
 'EBV EBNA1',
 'ECV11 PolyP',
 'ECV13 PolyP',
 'ECV14 PolyP',
 'ECV25 PolyP',
 'ECV3 PolyP',
 'ECV30 PolyP',
 'ECV9 PolyP',
 'EMCV PolyP',
 'EV-A71 PolyP',
 'EV-C VP1',
 'EV-D68 VP1',
 'G.haemolysans FnBPB',
 'GAD65',
 'H. influenzae PBPs',
 'H.pylori TonB',
 'HAdV-A SSB',
 'HAdV-B pIIIa',
 'HAdV-C pIII',
 'HAdV-D pIIIa',
 'HBV Pol',
 'HBoV2c-PK NS1',
 'HBoV3 NS1',
 'HBoV4-NI NS1',
 'HCMV UL32',
 'HCoV-229E NC',
 'HCoV-HKU1 NC',
 'HCoV-NL63 NC',
 'HCoV-OC43 NC',
 'HCoV